# 05 — Experiments

**Workstream E, task E1.** Runs all eight strategies on all 100 experiments and builds the **raw results
frame**: one row per experiment × strategy, every metric before costs, after costs, and after 2× costs.
Notebook 06 summarises this frame, and notebook 07 re-cuts it for robustness.

| Task | Output |
| --- | --- |
| E1 | `results`: 800 rows (100 experiments × 8 strategies), columns (basis, metric) with basis gross / net / net_2x |
| E10 | The optimisation-failure log, Table A.1's source |

Nothing is written to disk. The grid takes a few seconds, so notebooks 06 and 07 rebuild it from the seed
in the same way, and this notebook shows that the rebuild is exact.

Engine: `src/backtest.py` (notebook 03). Strategies: `src/strategies.py` (notebook 04). Metrics:
`src/performance.py`, defined in Table 4.1 (notebook 06).

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import backtest as bt
from src import config as cfg
from src import data, fmt, viz
from src import performance as pf
from src import strategies as st

viz.use_house_style()
np.random.seed(cfg.SEED)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)

---

## E1 — The grid

Every strategy runs on the same 100 (window, subset) pairs, under the same engine, constraints and cost
model. The 2× cost runs make the same trades as the 1× runs and differ only in the costs deducted, so
the 2× column is ready for workstream F without a second design.

In [2]:
close, volume, extraction_date = data.download_raw()
panel = data.build_panel(close, volume, extraction_date)
prices = panel.prices[cfg.ASSET_ORDER]
experiments = bt.sample_experiments(prices)

grid = pf.run_grid(prices, experiments, st.STRATEGIES, panel.risk_free)
results = pf.results_frame(grid, panel.risk_free)

print(f"Extraction date : {panel.extraction_date.isoformat()}")
print(f"Seed            : {cfg.SEED}")
print(f"Runs            : {len(grid.runs):,} at 1x costs + {len(grid.stressed):,} at 2x")

Extraction date : 2026-10-07
Seed            : 20261001
Runs            : 800 at 1x costs + 800 at 2x


---

## The raw results frame

Values are decimals (ratios unitless). Percent conversion happens only when a table is displayed.

In [3]:
print(f"results: {results.shape[0]} rows x {results.shape[1]} columns")
print("Bases  :", list(results.columns.levels[0]))
print("Metrics:", list(results.columns.levels[1]))
display(results["net"].head(16).round(4))

results: 800 rows x 24 columns
Bases  : ['gross', 'net', 'net_2x']
Metrics: ['ann_return', 'ann_vol', 'calmar', 'max_dd', 'sharpe', 'sortino', 'sterling', 'turnover']


metric               ann_return  ann_vol  sharpe  sortino  max_dd  calmar  sterling  turnover
experiment strategy                                                                          
1          EW            0.2267   0.2114  0.8788   1.4158 -0.1981  1.1444    0.7605    0.2605
           GMV          -0.0195   0.0761 -0.7565  -1.1217 -0.0948 -0.2052   -0.0999    0.4149
           MV            0.0660   0.2038  0.2149   0.3102 -0.2120  0.3112    0.2115    1.4317
           MSR           0.2260   0.3381  0.6490   1.0331 -0.3055  0.7396    0.5572    1.9130
           IV            0.0710   0.1031  0.3210   0.5094 -0.1112  0.6385    0.3362    0.3260
           ERC           0.0692   0.1016  0.3079   0.4882 -0.1101  0.6288    0.3295    0.3371
           MDP           0.0671   0.0995  0.2926   0.4637 -0.1103  0.6085    0.3192    0.4366
           MDC           0.2284   0.2037  0.9109   1.4749 -0.1896  1.2047    0.7887    0.3940
2          EW            0.5568   0.1477  2.7166   4.5587 -0.1320  4.2190    2.4002    0.2979
           GMV           0.2982   0.1178  1.8295   2.8520 -0.0949  3.1433    1.5302    0.3881
           MV            1.0341   0.2876  2.4331   3.9060 -0.1639  6.3094    3.9186    0.3359
           MSR           0.8512   0.2112  2.7761   4.4755 -0.1747  4.8731    3.0989    1.1746
           IV            0.4251   0.1263  2.4531   4.0702 -0.1172  3.6258    1.9567    0.3249
           ERC           0.4305   0.1255  2.4997   4.1668 -0.1149  3.7480    2.0037    0.2972
           MDP           0.4460   0.1251  2.5926   4.3660 -0.1118  3.9899    2.1059    0.2794
           MDC           0.5929   0.1482  2.8638   4.8671 -0.1273  4.6579    2.6086    0.3094

In [4]:
# Integrity of the frame, asserted.
# 1. Complete: every experiment x strategy, nothing dropped, nothing missing.
assert len(results) == cfg.N_EXPERIMENTS * len(cfg.STRATEGY_ORDER)
assert results.notna().all().all(), results.columns[results.isna().any()].tolist()

# 2. Every run is a full evaluation year.
assert all(len(r.net) == cfg.TRADING_DAYS for r in grid.runs.values())

# 3. Costs only ever subtract, and 2x subtracts more, in every single run.
assert (results[("net", "ann_return")] <= results[("gross", "ann_return")] + 1e-15).all()
assert (results[("net_2x", "ann_return")] <= results[("net", "ann_return")] + 1e-15).all()

# 4. Turnover is a property of the trades, not of the cost level.
assert all(grid.runs[k].turnover == grid.stressed[k].turnover for k in grid.runs)

# 5. Reproducible: rebuilding from the seed gives the identical frame, to the last bit.
rebuilt = pf.results_frame(pf.run_grid(prices, bt.sample_experiments(prices), st.STRATEGIES, panel.risk_free),
                           panel.risk_free)
pd.testing.assert_frame_equal(rebuilt, results, check_exact=True)

print("Raw results frame: complete, consistent across cost bases, and reproducible under the seed.")

Raw results frame: complete, consistent across cost bases, and reproducible under the seed.


---

## E10 — Optimisation failures

Every failure from every run, 1× and 2× (which fail identically, because costs never affect the weights).
The appendix table uses the 1× runs.

In [5]:
failures = bt.failure_log(list(grid.runs.values()))
failures_2x = bt.failure_log(list(grid.stressed.values()))
assert failures.equals(failures_2x)

counts = failures.groupby("strategy").size().reindex(cfg.STRATEGY_ORDER, fill_value=0)
print(f"Failures: {len(failures)} of {len(grid.runs) * len(bt.Schedule().execution_positions()):,} "
      "rebalances (1x runs). By strategy:", counts.to_dict())
display(failures.assign(date=failures["date"].dt.date))

Failures: 7 of 3,200 rebalances (1x runs). By strategy: {'EW': 0, 'GMV': 0, 'MV': 0, 'MSR': 7, 'IV': 0, 'ERC': 0, 'MDP': 0, 'MDC': 0}


,experiment,date,strategy,error,fallback
0,1,2022-09-30,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.00%/yr); tangency portfolio undefined,Held drifted weights (no trade)
1,1,2022-12-30,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.99%/yr); tangency portfolio undefined,Held drifted weights (no trade)
2,16,2019-01-07,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.93%/yr); tangency portfolio undefined,Held drifted weights (no trade)
3,49,2022-09-23,MSR,InfeasibleError: no asset has expected return above the risk-free rate (0.94%/yr); tangency portfolio undefined,Held drifted weights (no trade)
4,49,2022-12-22,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.90%/yr); tangency portfolio undefined,Held drifted weights (no trade)
5,88,2022-05-13,MSR,InfeasibleError: no asset has expected return above the risk-free rate (0.19%/yr); tangency portfolio undefined,Held drifted weights (no trade)
6,92,2018-11-19,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.81%/yr); tangency portfolio undefined,Held drifted weights (no trade)


---

## Handoff

```python
grid = pf.run_grid(prices, experiments, st.STRATEGIES, panel.risk_free)
results = pf.results_frame(grid, panel.risk_free)     # identical in 06 and 07
```

| Object | Used by |
| --- | --- |
| `results[("net", ...)]` | 06: Tables 4.2–4.3, Figures 4.1–4.2 |
| `results[("gross", ...)]` | 06: Table 4.4, appendix Table A4.1 |
| `results[("net_2x", ...)]` | 07: Table 5.1 |
| `grid.runs[(experiment, strategy)]` | 06: Figures 4.3–4.4 (daily wealth and weights); 07: concentration |
| `failures` | 06: Table A.1 |